# 🛡️ AE-WGAN NSL-KDD — Reproduction fidèle du code original

**Ce notebook reproduit exactement l'architecture du code GitHub des auteurs.**

### Différences avec notre implémentation précédente :
1. **BiGAN hybride** : le Discriminator reçoit `concatenate([z, Flatten(image)])` — pas juste l'image
2. **Generator 5 couches** : 20→40→80→120→160→24 (au lieu de 40→80→120→24)
3. **Encoder latent_dim=32** : sortie dimension 32 sans activation (au lieu de 16 avec relu)
4. **Reshape 4×6** : les 24 features sont traitées comme une image 4×6×1
5. **Activation relu** partout dans le Generator (pas sigmoid en sortie)
6. **Ratio 1:1** : Discriminator et Generator entraînés 1 fois chacun par epoch (pas 5:1)
7. **Pas de gradient penalty** : simple Wasserstein loss `K.mean(y_true * y_pred)`
8. **Discriminator sigmoid** en sortie (pas linéaire)
9. **Pas de Denoising** : l'Encoder encode directement, pas de bruit ajouté

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q tensorflow numpy pandas scikit-learn matplotlib scipy

In [ ]:
import os, urllib.request, time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import (Input, Dense, Reshape, Flatten,
                                      Concatenate, BatchNormalization)
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K

from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score)
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import cdist
from tensorflow.keras import layers

import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

## 1. Preprocessing NSL-KDD

In [ ]:
url = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/"
files = {"KDDTrain+.txt": url + "KDDTrain%2B.txt", "KDDTest+.txt": url + "KDDTest%2B.txt"}
for fname, furl in files.items():
    if not os.path.exists(fname):
        urllib.request.urlretrieve(furl, fname)
        print(f"✅ {fname}")

NSL_KDD_COLUMNS = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root',
    'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds',
    'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
    'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate',
    'label', 'difficulty_level'
]

NSL_KDD_ATTACK_MAP = {
    'normal': 'Normal',
    'back': 'DoS', 'land': 'DoS', 'neptune': 'DoS', 'pod': 'DoS',
    'smurf': 'DoS', 'teardrop': 'DoS', 'mailbomb': 'DoS', 'apache2': 'DoS',
    'processtable': 'DoS', 'udpstorm': 'DoS',
    'satan': 'Probe', 'ipsweep': 'Probe', 'nmap': 'Probe', 'portsweep': 'Probe',
    'mscan': 'Probe', 'saint': 'Probe',
    'guess_passwd': 'R2L', 'ftp_write': 'R2L', 'imap': 'R2L', 'phf': 'R2L',
    'multihop': 'R2L', 'warezmaster': 'R2L', 'warezclient': 'R2L', 'spy': 'R2L',
    'xlock': 'R2L', 'xsnoop': 'R2L', 'snmpguess': 'R2L', 'snmpgetattack': 'R2L',
    'httptunnel': 'R2L', 'sendmail': 'R2L', 'named': 'R2L', 'worm': 'R2L',
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'rootkit': 'U2R',
    'perl': 'U2R', 'sqlattack': 'U2R', 'xterm': 'U2R', 'ps': 'U2R',
}

# Charger
df_train = pd.read_csv("KDDTrain+.txt", header=None, names=NSL_KDD_COLUMNS)
df_test = pd.read_csv("KDDTest+.txt", header=None, names=NSL_KDD_COLUMNS)
df_train = df_train.drop(columns=['difficulty_level'])
df_test = df_test.drop(columns=['difficulty_level'])
df_train['label'] = df_train['label'].str.strip().str.rstrip('.')
df_test['label'] = df_test['label'].str.strip().str.rstrip('.')

y_train_cat = df_train['label'].map(NSL_KDD_ATTACK_MAP).fillna('Unknown')
y_test_cat = df_test['label'].map(NSL_KDD_ATTACK_MAP).fillna('Unknown')
y_train_detail = df_train['label'].copy()

X_train_df = df_train.drop(columns=['label'])
X_test_df = df_test.drop(columns=['label'])

# Data Cleaning
removed = []
for col in X_train_df.columns:
    if X_train_df[col].nunique() <= 1:
        removed.append(col); continue
    if X_train_df[col].value_counts(normalize=True).iloc[0] >= 0.99:
        removed.append(col)
X_train_df = X_train_df.drop(columns=removed)
X_test_df = X_test_df.drop(columns=removed, errors='ignore')

# Ordinal Encoding
cat_cols = X_train_df.select_dtypes(include=['object']).columns.tolist()
enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train_df[cat_cols] = enc.fit_transform(X_train_df[cat_cols])
X_test_df[cat_cols] = enc.transform(X_test_df[cat_cols])
feature_names = X_train_df.columns.tolist()

# Min-Max Scaling
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train_df.values)
X_test_scaled = scaler.transform(X_test_df.values)

# ANOVA → 24 features
le_a = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
y_enc = le_a.fit_transform(y_train_cat.values.reshape(-1, 1)).ravel()
selector = SelectKBest(score_func=f_classif, k=24)
X_train_final = selector.fit_transform(X_train_scaled, y_enc)
X_test_final = selector.transform(X_test_scaled)
INPUT_DIM = 24

print(f"✅ Preprocessing: Train {X_train_final.shape}, Test {X_test_final.shape}")
print(f"Distribution: {dict(y_train_cat.value_counts())}")

## 2. AE-WGAN — Reproduction exacte du code original GitHub

Architecture BiGAN hybride :
- **Encoder** : Flatten(4×6) → Dense(20, relu) → Dense(32, linear)
- **Generator** : Dense(20) → Dense(40) → Dense(80) → Dense(120) → Dense(160) → Dense(24) → Reshape(4,6,1) — tout relu
- **Discriminator** : concatenate([z(32), Flatten(img)(24)]) → Dense(80, relu) → Dense(40, relu) → Dense(1, sigmoid)
- **Loss** : Wasserstein loss `K.mean(y_true * y_pred)`
- **Optimizer** : Adamax(0.00009)
- **Training** : ratio 1:1 (pas 5:1), 101 epochs

In [ ]:
# ============================================================
# AE-WGAN — REPRODUCTION EXACTE DU CODE GITHUB
# ============================================================

class AEWGAN_Original:
    """
    Reproduction fidèle du code GitHub marafah/AE-WGAN.
    Architecture BiGAN: Discriminator reçoit (z, image) concaténés.
    """
    def __init__(self):
        self.img_rows = 4
        self.img_cols = 6
        self.channels = 1
        self.img_shape = (self.img_rows, self.img_cols, self.channels)
        self.latent_dim = 32  # Code original: 32

        optimizer = tf.keras.optimizers.Adamax(0.00009)

        # Build Discriminator
        self.discriminator = self._build_discriminator()
        self.discriminator.compile(
            loss=self._wasserstein_loss,
            optimizer=optimizer,
            metrics=['accuracy']
        )

        # Build Generator
        self.generator = self._build_generator()

        # Build Encoder
        self.encoder = self._build_encoder()

        # Combined model: trains Generator + Encoder to fool Discriminator
        self.discriminator.trainable = False

        # Generator path: z → Generator → fake_img
        z = Input(shape=(self.latent_dim,))
        img_ = self.generator(z)

        # Encoder path: real_img → Encoder → z_encoded
        img = Input(shape=self.img_shape)
        z_ = self.encoder(img)

        # Discriminator evaluates pairs:
        # (z_random, generated_img) → should be fake
        # (z_encoded, real_img) → should be valid
        fake = self.discriminator([z, img_])
        valid = self.discriminator([z_, img])

        # Combined model
        self.combined = Model([z, img], [fake, valid])
        self.combined.compile(
            loss=[self._wasserstein_loss, self._wasserstein_loss],
            optimizer=optimizer
        )

    def _wasserstein_loss(self, y_true, y_pred):
        """Exactement comme le code original."""
        return K.mean(y_true * y_pred)

    def _build_encoder(self):
        """
        Code original:
        Flatten(4,6,1) → Dense(20, relu) → Dense(32, no activation)
        """
        model = Sequential([
            Flatten(input_shape=self.img_shape),
            Dense(20, activation='relu'),
            Dense(self.latent_dim),  # Pas d'activation (linéaire)
        ])
        img = Input(shape=self.img_shape)
        z = model(img)
        return Model(img, z, name='encoder')

    def _build_generator(self):
        """
        Code original:
        Dense(20, relu) → Dense(40, relu) → Dense(80, relu)
        → Dense(120, relu) → Dense(160, relu) → Dense(24, relu)
        → Reshape(4,6,1)
        """
        model = Sequential([
            Dense(20, input_dim=self.latent_dim, activation='relu'),
            Dense(40, activation='relu'),
            Dense(80, activation='relu'),
            Dense(120, activation='relu'),
            Dense(160, activation='relu'),
            Dense(np.prod(self.img_shape), activation='relu'),
            Reshape(self.img_shape),
        ])
        z = Input(shape=(self.latent_dim,))
        gen_img = model(z)
        return Model(z, gen_img, name='generator')

    def _build_discriminator(self):
        """
        Code original:
        concatenate([z(32), Flatten(img)(24)]) = 56 dims
        → Dense(80, relu) → Dense(40, relu) → Dense(1, sigmoid)
        """
        z = Input(shape=(self.latent_dim,))
        img = Input(shape=self.img_shape)
        d_in = Concatenate()([z, Flatten()(img)])

        x = Dense(80, activation='relu')(d_in)
        x = Dense(40, activation='relu')(x)
        validity = Dense(1, activation='sigmoid')(x)

        return Model([z, img], validity, name='discriminator')

    def train(self, X_attack_2d, epochs=101, batch_size=32, verbose=1):
        """
        Training loop exactement comme le code original.
        Ratio 1:1 — pas de n_critic.
        """
        # Reshape en image 4×6×1 comme le code original
        reshaped = X_attack_2d.reshape(-1, self.img_rows, self.img_cols, self.channels)
        n_samples = reshaped.shape[0]

        if n_samples < batch_size:
            batch_size = max(n_samples // 2, 2)

        valid = np.ones((batch_size, 1))
        fake = np.zeros((batch_size, 1))

        history = {'d_loss': [], 'd_acc': [], 'g_loss': []}

        for epoch in range(epochs):
            # ---- Train Discriminator ----
            # Sample noise and generate fake images
            z = np.random.normal(0, 1, size=(batch_size, self.latent_dim))
            imgs_ = self.generator.predict(z, verbose=0)

            # Select random batch of real images and encode
            idx = np.random.randint(0, n_samples, batch_size)
            imgs = reshaped[idx]
            z_ = self.encoder.predict(imgs, verbose=0)

            # Train: (encoded_z, real_img) → valid, (random_z, gen_img) → fake
            d_loss_real = self.discriminator.train_on_batch([z_, imgs], valid)
            d_loss_fake = self.discriminator.train_on_batch([z, imgs_], fake)
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

            # ---- Train Generator + Encoder ----
            g_loss = self.combined.train_on_batch([z, imgs], [valid, fake])

            history['d_loss'].append(d_loss[0])
            history['d_acc'].append(d_loss[1])
            history['g_loss'].append(g_loss[0])

            if verbose and (epoch + 1) % max(1, epochs // 5) == 0:
                print(f"      Epoch {epoch+1}/{epochs} — "
                      f"D loss: {d_loss[0]:.4f}, D acc: {100*d_loss[1]:.1f}%, "
                      f"G loss: {g_loss[0]:.4f}")

        return history

    def generate(self, n):
        """
        Générer n samples. Reproduit la méthode generate_data du code original:
        bruit de shape (32, latent_dim), prendre gen_imgs[0].
        """
        generated = []
        for _ in range(n):
            noise = np.random.normal(size=(32, self.latent_dim))
            gen_imgs = self.generator.predict(noise, verbose=0)
            generated.append(gen_imgs[0])  # Prendre le premier comme le code original
        generated = np.array(generated)
        # Reshape de image (4,6,1) vers vecteur (24,)
        return generated.reshape(-1, self.img_rows * self.img_cols)

print("✅ AEWGAN_Original défini (reproduction exacte du GitHub)")

## 3. Oversampling avec le code original

In [ ]:
# ============================================================
# OVERSAMPLING — Code original
# Le code original inclut AUSSI 'normal' dans les types à générer
# ============================================================

print("=" * 60)
print("AE-WGAN OVERSAMPLING (architecture originale GitHub)")
print("=" * 60)

# Code original: génère pour TOUS les types y compris normal
all_attack_types = sorted(y_train_detail.unique())
print(f"Types à traiter: {len(all_attack_types)}")

MIN_SAMPLES = 20
EPOCHS = 101
BATCH_SIZE = 32

all_synthetic_X = []
all_synthetic_y = []
trained_models = {}

start_time = time.time()

for attack_name in all_attack_types:
    mask = y_train_detail == attack_name
    X_attack = X_train_final[mask]
    n_orig = X_attack.shape[0]

    if n_orig < MIN_SAMPLES:
        print(f"\n  [{attack_name}] ⏭️ Sauté ({n_orig} < {MIN_SAMPLES})")
        continue

    print(f"\n  [{attack_name}] 🧠 Training ({n_orig} samples)...")

    model = AEWGAN_Original()

    bs = min(BATCH_SIZE, max(n_orig // 2, 2))
    model.train(X_attack, epochs=EPOCHS, batch_size=bs, verbose=1)

    # Code original: génère N + 20000 samples
    n_to_generate = n_orig + 20000
    synthetic = model.generate(n_to_generate)

    all_synthetic_X.append(synthetic)
    all_synthetic_y.extend([attack_name] * n_to_generate)
    trained_models[attack_name] = model

    print(f"  [{attack_name}] ✅ Généré {n_to_generate} samples")

elapsed = (time.time() - start_time) / 60
print(f"\n--- {elapsed:.1f} minutes ---")

# Combiner
X_synthetic_all = np.vstack(all_synthetic_X)
y_synthetic_all = pd.Series(all_synthetic_y)
X_augmented = np.vstack([X_train_final, X_synthetic_all])
y_augmented_detail = pd.concat([y_train_detail, y_synthetic_all], ignore_index=True)
y_augmented_cat = y_augmented_detail.map(NSL_KDD_ATTACK_MAP).fillna('Unknown')

print(f"\n{'='*60}")
print(f"  Original:    {X_train_final.shape[0]}")
print(f"  Synthétique: {X_synthetic_all.shape[0]}")
print(f"  Total:       {X_augmented.shape[0]}")
print(f"{'='*60}")

## 4. Évaluation qualité générative

In [ ]:
# ============================================================
# GENERATIVE QUALITY
# ============================================================

def compute_cosine_sim(X_real, X_synth):
    real_mean = X_real.mean(axis=0).reshape(1, -1)
    synth_mean = X_synth.mean(axis=0).reshape(1, -1)
    return cosine_similarity(real_mean, synth_mean)[0, 0]

def discrimination_test(X_real, X_synth):
    n = min(len(X_real), len(X_synth), 5000)
    X_r = X_real[np.random.choice(len(X_real), n, replace=n > len(X_real))]
    X_s = X_synth[np.random.choice(len(X_synth), n, replace=n > len(X_synth))]
    X_c = np.vstack([X_r, X_s])
    y_c = np.array([1]*n + [0]*n)
    X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.3, random_state=42, stratify=y_c)
    clf = RandomForestClassifier(100, random_state=42, n_jobs=-1)
    clf.fit(X_tr, y_tr)
    return accuracy_score(y_te, clf.predict(X_te))

print(f"{'─'*70}")
print(f"  {'Attaque':<20} {'N orig':>7} {'N gen':>7} {'Cos Sim':>8} {'Disc Acc':>10} {'Qualité':>10}")
print(f"{'─'*70}")

for attack_name, model in trained_models.items():
    mask = y_train_detail == attack_name
    X_real = X_train_final[mask]
    if len(X_real) < 20:
        continue
    X_synth = model.generate(min(len(X_real), 500))

    cos_s = compute_cosine_sim(X_real, X_synth)
    disc = discrimination_test(X_real, X_synth)

    quality = "Excellent" if disc < 0.55 else "Good" if disc < 0.65 else "Fair" if disc < 0.75 else "Poor"

    print(f"  {attack_name:<20} {len(X_real):>7} {len(X_real)+20000:>7} {cos_s:>8.4f} {disc:>9.2%} {quality:>10}")

print(f"{'─'*70}")

## 5. Classifieurs (Table 4 du paper — NSL-KDD)

In [ ]:
# ============================================================
# CLASSIFIEURS
# ============================================================

def build_dt(n_classes=2):
    return DecisionTreeClassifier(max_depth=7 if n_classes==2 else 9, random_state=42)

def build_rf(n_classes=2):
    return RandomForestClassifier(100, max_depth=8 if n_classes==2 else 10, random_state=42, n_jobs=-1)

def build_cnn(input_dim, n_classes=2):
    out = 1 if n_classes==2 else n_classes
    act = 'sigmoid' if n_classes==2 else 'softmax'
    m = Sequential([
        layers.Input(shape=(input_dim, 1)),
        layers.Conv1D(40, 3, activation='relu', padding='same'),
        layers.Dropout(0.1), layers.Flatten(),
        layers.Dense(40, activation='relu'),
        layers.Dense(out, activation=act)
    ], name='CNN')
    loss = 'binary_crossentropy' if n_classes==2 else 'sparse_categorical_crossentropy'
    m.compile(optimizer=Adam(1e-3), loss=loss, metrics=['accuracy'])
    return m

def build_gru(input_dim, n_classes=2):
    out = 1 if n_classes==2 else n_classes
    act = 'sigmoid' if n_classes==2 else 'softmax'
    m = Sequential(name='GRU')
    m.add(layers.Input(shape=(input_dim, 1)))
    if n_classes==2:
        m.add(layers.GRU(30, activation='relu'))
        m.add(layers.Dropout(0.1))
        m.add(layers.Dense(10, activation='relu'))
        m.add(layers.Dense(10, activation='relu'))
    else:
        m.add(layers.GRU(30, activation='relu', return_sequences=True))
        m.add(layers.Dropout(0.1))
        m.add(layers.GRU(30, activation='relu'))
        m.add(layers.Dropout(0.1))
        m.add(layers.Dense(30, activation='relu'))
        m.add(layers.Dropout(0.1))
        m.add(layers.Dense(30, activation='relu'))
    m.add(layers.Dense(out, activation=act))
    loss = 'binary_crossentropy' if n_classes==2 else 'sparse_categorical_crossentropy'
    m.compile(optimizer=Adam(1e-3), loss=loss, metrics=['accuracy'])
    return m

def build_lstm(input_dim, n_classes=2):
    out = 1 if n_classes==2 else n_classes
    act = 'sigmoid' if n_classes==2 else 'softmax'
    m = Sequential(name='LSTM')
    m.add(layers.Input(shape=(input_dim, 1)))
    if n_classes==2:
        m.add(layers.LSTM(10, activation='relu', return_sequences=True))
        m.add(layers.LSTM(20, activation='relu'))
        m.add(layers.Dropout(0.5))
        m.add(layers.Dense(10, activation='relu'))
    else:
        m.add(layers.LSTM(10, activation='relu', return_sequences=True))
        m.add(layers.Dropout(0.1))
        m.add(layers.LSTM(20, activation='relu', return_sequences=True))
        m.add(layers.Dropout(0.2))
        m.add(layers.LSTM(20, activation='relu'))
        m.add(layers.Dropout(0.1))
        m.add(layers.Dense(10, activation='relu'))
    m.add(layers.Dense(out, activation=act))
    loss = 'binary_crossentropy' if n_classes==2 else 'sparse_categorical_crossentropy'
    m.compile(optimizer=Adam(1e-3), loss=loss, metrics=['accuracy'])
    return m

def build_rnn(input_dim, n_classes=2):
    out = 1 if n_classes==2 else n_classes
    act = 'sigmoid' if n_classes==2 else 'softmax'
    m = Sequential(name='RNN')
    m.add(layers.Input(shape=(input_dim, 1)))
    if n_classes==2:
        m.add(layers.SimpleRNN(30, activation='relu'))
        m.add(layers.Dropout(0.1))
        m.add(layers.Dense(10, activation='relu'))
        m.add(layers.Dense(10, activation='relu'))
    else:
        m.add(layers.SimpleRNN(50, activation='relu', return_sequences=True))
        m.add(layers.Dropout(0.2))
        m.add(layers.SimpleRNN(50, activation='relu', return_sequences=True))
        m.add(layers.Dropout(0.2))
        m.add(layers.SimpleRNN(50, activation='relu', return_sequences=True))
        m.add(layers.Dropout(0.2))
        m.add(layers.SimpleRNN(50, activation='relu'))
        m.add(layers.Dropout(0.2))
    m.add(layers.Dense(out, activation=act))
    loss = 'binary_crossentropy' if n_classes==2 else 'sparse_categorical_crossentropy'
    m.compile(optimizer=Adam(1e-3), loss=loss, metrics=['accuracy'])
    return m

print("✅ Classifieurs définis")

## 6. Évaluation Baseline + AE-WGAN

In [ ]:
# ============================================================
# EVALUATION FUNCTIONS
# ============================================================

def prepare_labels(y_train, y_test, binary=True):
    le = LabelEncoder()
    if binary:
        return (y_train != 'Normal').astype(int).values, (y_test != 'Normal').astype(int).values, le
    else:
        le.fit(list(set(y_train.unique()) | set(y_test.unique())))
        return le.transform(y_train), le.transform(y_test), le

def train_and_eval(name, clf, X_tr, y_tr, X_te, y_te, epochs=50, bs=32):
    is_dl = name in ['CNN', 'GRU', 'LSTM', 'RNN']
    if is_dl:
        X_tr_3d = X_tr.reshape(X_tr.shape[0], X_tr.shape[1], 1)
        X_te_3d = X_te.reshape(X_te.shape[0], X_te.shape[1], 1)
        es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        clf.fit(X_tr_3d, y_tr, epochs=epochs, batch_size=bs,
                validation_split=0.1, callbacks=[es], verbose=0)
        y_prob = clf.predict(X_te_3d, verbose=0)
        y_pred = (y_prob.flatten()>0.5).astype(int) if y_prob.shape[-1]==1 else np.argmax(y_prob, axis=1)
    else:
        clf.fit(X_tr, y_tr)
        y_pred = clf.predict(X_te)
    avg = 'binary' if len(np.unique(y_te))==2 else 'weighted'
    return {
        'classifier': name,
        'accuracy': accuracy_score(y_te, y_pred),
        'precision': precision_score(y_te, y_pred, average=avg, zero_division=0),
        'recall': recall_score(y_te, y_pred, average=avg, zero_division=0),
        'f1_score': f1_score(y_te, y_pred, average=avg, zero_division=0),
    }

def run_all_classifiers(X_tr, y_tr_cat, X_te, y_te_cat, input_dim, binary=True, epochs=50, bs=32):
    label = "BINAIRE" if binary else "MULTICLASSE"
    print(f"\n{'='*60}")
    print(f"  CLASSIFICATION {label}")
    print(f"{'='*60}")
    y_tr, y_te, le = prepare_labels(y_tr_cat, y_te_cat, binary)
    n_classes = 2 if binary else len(np.unique(y_tr))
    print(f"  Classes: {n_classes}, Train: {X_tr.shape[0]}, Test: {X_te.shape[0]}")
    builders = {
        'DT': lambda: build_dt(n_classes),
        'RF': lambda: build_rf(n_classes),
        'CNN': lambda: build_cnn(input_dim, n_classes),
        'GRU': lambda: build_gru(input_dim, n_classes),
        'LSTM': lambda: build_lstm(input_dim, n_classes),
        'RNN': lambda: build_rnn(input_dim, n_classes),
    }
    results = []
    for name, builder in builders.items():
        print(f"  Training {name}...", end=" ")
        r = train_and_eval(name, builder(), X_tr, y_tr, X_te, y_te, epochs, bs)
        print(f"Acc={r['accuracy']:.4f} F1={r['f1_score']:.4f}")
        results.append(r)
    return results

print("✅ Fonctions d'évaluation prêtes")

In [ ]:
# ============================================================
# RUN CLASSIFICATION
# ============================================================

print("🔄 Baseline...")
baseline_binary = run_all_classifiers(
    X_train_final, y_train_cat, X_test_final, y_test_cat, INPUT_DIM, binary=True)
baseline_multi = run_all_classifiers(
    X_train_final, y_train_cat, X_test_final, y_test_cat, INPUT_DIM, binary=False)

print("\n🔄 AE-WGAN (architecture originale)...")
enhanced_binary = run_all_classifiers(
    X_augmented, y_augmented_cat, X_test_final, y_test_cat, INPUT_DIM, binary=True)
enhanced_multi = run_all_classifiers(
    X_augmented, y_augmented_cat, X_test_final, y_test_cat, INPUT_DIM, binary=False)

## 7. Comparaison et Export

In [ ]:
# ============================================================
# COMPARISON
# ============================================================

print("\n" + "=" * 70)
print("  COMPARAISON BASELINE vs AE-WGAN (architecture originale GitHub)")
print("=" * 70)

for task, base_res, enh_res in [
    ("BINAIRE", baseline_binary, enhanced_binary),
    ("MULTICLASSE", baseline_multi, enhanced_multi)
]:
    print(f"\n  --- {task} ---")
    print(f"  {'Clf':<8} {'Base Acc':>9} {'AEWGAN Acc':>11} {'Δ Acc':>7} "
          f"{'Base F1':>9} {'AEWGAN F1':>10} {'Δ F1':>7}")
    print(f"  {'─'*63}")
    for b, e in zip(base_res, enh_res):
        d_acc = e['accuracy'] - b['accuracy']
        d_f1 = e['f1_score'] - b['f1_score']
        sa = '+' if d_acc >= 0 else ''
        sf = '+' if d_f1 >= 0 else ''
        print(f"  {b['classifier']:<8} {b['accuracy']:>9.4f} {e['accuracy']:>11.4f} "
              f"{sa}{d_acc:>6.4f} {b['f1_score']:>9.4f} {e['f1_score']:>10.4f} "
              f"{sf}{d_f1:>6.4f}")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('NSL-KDD: Baseline vs AE-WGAN (Architecture originale)', fontsize=16, fontweight='bold')
metrics = ['accuracy', 'precision', 'recall', 'f1_score']
titles = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
for idx, (metric, title) in enumerate(zip(metrics, titles)):
    ax = axes[idx//2][idx%2]
    classifiers = [r['classifier'] for r in baseline_binary]
    x = np.arange(len(classifiers))
    w = 0.2
    ax.bar(x-1.5*w, [r[metric] for r in baseline_binary], w, label='Baseline Bin', color='#3498db', alpha=0.8)
    ax.bar(x-0.5*w, [r[metric] for r in enhanced_binary], w, label='AE-WGAN Bin', color='#e74c3c', alpha=0.8)
    ax.bar(x+0.5*w, [r[metric] for r in baseline_multi], w, label='Baseline Multi', color='#2ecc71', alpha=0.8)
    ax.bar(x+1.5*w, [r[metric] for r in enhanced_multi], w, label='AE-WGAN Multi', color='#f39c12', alpha=0.8)
    ax.set_xlabel('Classifieur'); ax.set_ylabel(title); ax.set_title(title)
    ax.set_xticks(x); ax.set_xticklabels(classifiers)
    ax.legend(fontsize=8); ax.set_ylim(0.5, 1.05); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results_original_architecture.png', dpi=150, bbox_inches='tight')
plt.show()

# CSV
all_results = []
for r in baseline_binary: all_results.append({**r, 'type': 'Baseline', 'classification': 'Binary'})
for r in enhanced_binary: all_results.append({**r, 'type': 'AE-WGAN', 'classification': 'Binary'})
for r in baseline_multi: all_results.append({**r, 'type': 'Baseline', 'classification': 'Multiclass'})
for r in enhanced_multi: all_results.append({**r, 'type': 'AE-WGAN', 'classification': 'Multiclass'})
df_results = pd.DataFrame(all_results)
df_results.to_csv('results_original_architecture.csv', index=False)
print("\n✅ Résultats sauvegardés: results_original_architecture.csv")
print(df_results.to_string(index=False))

In [ ]:
# Télécharger les résultats
from google.colab import files
files.download('results_original_architecture.csv')
files.download('results_original_architecture.png')